In [1]:
import os
import torch
from torch import nn
import torch.optim as optim
import time
from tqdm import tqdm
import datasets
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import LoraConfig, get_peft_model
# from transformers import BitsAndBytesConfig
import open_clip
import torch.nn.utils.rnn as rnn_utils
from argparse import ArgumentParser
from peft import PeftModel
import torchvision.transforms.functional as TF

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


class CLIP_Adapted_ImageEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip_model = clip_model

        self.adapter = nn.Sequential(
            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512)
        )

    def forward(self, image):
        return self.adapter(self.clip_model.encode_image(image))

/home/mmuuser/anaconda3/envs/vlunet/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


/home/mmuuser/anaconda3/envs/vlunet/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
class CLIP_Adapted_TextEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip_model = clip_model
        self.adapter = nn.Sequential(
            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512),

            nn.LeakyReLU(),

            nn.Linear(512, 512),
            nn.LayerNorm(512)
        )

    def forward(self, text):
        return self.adapter(self.clip_model.encode_text(text))

class DA_adapter(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        for param in clip_model.parameters():
            param.requires_grad = False
        self.ad_imageEncoder = CLIP_Adapted_ImageEncoder(clip_model=clip_model)
        self.ad_textEncoder = CLIP_Adapted_TextEncoder(clip_model=clip_model)

    def forward(self, image, text):
        image = self.ad_imageEncoder(image)
        text = self.ad_textEncoder(text)
        return image, text

In [3]:
parser = ArgumentParser(description='LDUN')
parser.add_argument('--about', type=str, default='5task')
parser.add_argument('--start_epoch', type=int, default=0, help='epoch number of start training')
parser.add_argument('--end_epoch', type=int, default=100, help='epoch number of end training')
parser.add_argument('--learning_rate', type=float, default=1e-5, help='learning rate')
parser.add_argument('--resume', type=bool, default=False, help='is resume')
parser.add_argument('--group_num', type=int, default=1, help='group number for training')
parser.add_argument('--gpu_list', type=str, default='1', help='gpu index')
parser.add_argument('--checkpoints_dir', type=str, default='ckpt/Phase3/BLIP_CLIP_degradation_extractor_Classifier/Checkpoint', help='checkpoints dir')
parser.add_argument('--log_dir', type=str, default='log', help='log directory')
parser.add_argument('--ext', type=str, default='.png', help='training data directory')
parser.add_argument('--is_aug', type=bool,default=False, help='is aug')
parser.add_argument('--is_clip_tuning', type=bool,default=False, help='is finetuning clip')
parser.add_argument('--patch_size', type=int, default=128, help='patchsize of input.')

# Noise, Haze, Rain, Blurr, Lowlight
NHRBL = ["./datasets/denoising_datasets/15_train_paths.txt",
 "./datasets/denoising_datasets/25_train_paths.txt",
 "./datasets/denoising_datasets/50_train_paths.txt",
 "./datasets/dehazing_datasets/train_paths.txt",
 "./datasets/deraining_datasets/Rain100L/train_paths.txt",
 "./datasets/deblurring_datasets/GoPro/train_paths.txt",
 "./datasets/delowlight_datasets/LoL/train_paths.txt"]

In [4]:
args = parser.parse_args(args=[])
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu_list
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda:0


In [5]:
model, preprocess, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
tokenizer = open_clip.get_tokenizer('ViT-B-32')

model = DA_adapter(model)
model.to(device)

DA_adapter(
  (ad_imageEncoder): CLIP_Adapted_ImageEncoder(
    (clip_model): CLIP(
      (visual): VisionTransformer(
        (patchnorm_pre_ln): Identity()
        (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
        (patch_dropout): Identity()
        (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (transformer): Transformer(
          (resblocks): ModuleList(
            (0-11): 12 x ResidualAttentionBlock(
              (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
              (attn): MultiheadAttention(
                (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
              )
              (ls_1): Identity()
              (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
              (mlp): Sequential(
                (c_fc): Linear(in_features=768, out_features=3072, bias=True)
                (gelu): GELU(approxima

In [6]:
from torchvision import transforms, models

class SimpleCNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.model = models.resnet18(pretrained=True)
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [7]:
DEGRADATIONS = ["haze", "noise", "rain", "lowlight", "blur"]
SEVERITIES = [1, 2, 3]

LABEL_MAP = {
    f"{deg}_{sev}": i
    for i, (deg, sev) in enumerate(
        [(d, s) for d in DEGRADATIONS for s in SEVERITIES]
    )
}

NUM_CLASSES = len(LABEL_MAP)

In [8]:
SAVE_DIR = "ckpt/Phase1/DegradationClassifier"
os.makedirs(SAVE_DIR, exist_ok=True)

LATEST_CKPT = os.path.join(SAVE_DIR, "latest_checkpoint.pth")

BATCH_SIZE = 100
EPOCHS = 10
LR = 1e-4

In [9]:
CKPT_PATH = "ckpt/Phase1/DegradationClassifier/best_degradation_classifier_model.pth"

ckpt = torch.load(CKPT_PATH, map_location=device)

LABEL_MAP = ckpt["label_map"]

IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}

In [10]:
checkpoint = torch.load(CKPT_PATH, map_location=device)

classifier_model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
classifier_model.load_state_dict(checkpoint["model_state_dict"])
classifier_model.eval()

print("✓ Classifier loaded")

✓ Classifier loaded


/home/mmuuser/anaconda3/envs/vlunet/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mmuuser/anaconda3/envs/vlunet/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [11]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

severity_map = {
    "1": "light",
    "2": "moderate",
    "3": "heavy"
}

In [19]:
@torch.inference_mode()
def generate_caption_batch_with_degradation(
    image_paths,
    blip_model,
    processor,
    classifier_model,
    device
):

    # =========================
    # 0. LOAD IMAGES
    # =========================
    images = []
    valid_paths = []

    for p in image_paths:
        try:
            img = Image.open(p).convert("RGB")
            images.append(img)
            valid_paths.append(p)
        except Exception as e:
            print(f"Failed to load {p}: {e}")

    if len(images) == 0:
        return []

    # =========================
    # 1. BLIP CAPTION (BATCH)
    # =========================
    prompts = [
        "Question: Describe this scene. Answer:"
        for _ in range(len(images))
    ]

    inputs = processor(
        images=images,
        text=prompts,
        padding=True,
        return_tensors="pt"
    ).to(device)

    # print("\n===== BLIP DEBUG =====")
    # print("Num images:", len(images))

    # if "pixel_values" in inputs:
    #     print("pixel_values:", inputs["pixel_values"].shape)

    # if "input_ids" in inputs:
    #     print("input_ids:", inputs["input_ids"].shape)

    generated_ids = blip_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        num_beams=1
    )

    # print("generated_ids shape:", generated_ids.shape)

    captions = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    captions = [
        c.replace("Question: Describe this scene. Answer:", "").strip()
        for c in captions
    ]

    # print("captions generated:", len(captions))

    # =========================
    # 2. CLASSIFIER (BATCHED)
    # =========================
    img_tensors = torch.stack([
        transform(img)
        for img in images
    ]).to(device)

    logits = classifier_model(img_tensors)

    pred_idxs = logits.argmax(dim=1).tolist()

    batch_degradation = []

    for pred_idx in pred_idxs:

        raw_label = IDX_TO_LABEL[pred_idx]

        deg_type, sev_num = raw_label.split("_")

        degradation_text = (
            f"{severity_map[sev_num]} "
            f"{deg_type.lower()}"
        )

        batch_degradation.append(degradation_text)

    # print("degradation labels:", len(batch_degradation))

    # =========================
    # 3. FUSION
    # =========================
    enhanced_captions = []

    min_len = min(
        len(captions),
        len(batch_degradation),
        len(valid_paths)
    )

    for i in range(min_len):

        cap = captions[i].rstrip(" .,!?:;")
        deg = batch_degradation[i]

        enhanced_captions.append(
            (
                valid_paths[i],
                f"{cap} in {deg} conditions"
            )
        )

    # print("enhanced captions:", len(enhanced_captions))
    # print("======================\n")

    return enhanced_captions

In [13]:
BASE_DIR = "datasets"

def fix_path(p):
    return os.path.join(BASE_DIR, p.replace("./", "").strip())

all_image_paths = []
missing_files = []

print("🔍 Building dataset paths...")

for file in tqdm(NHRBL, desc="📂 Dataset Files"):

    with open(file, "r") as f:
        lines = [l.strip() for l in f if l.strip()]

    for line in lines:

        if "," not in line:
            continue

        noisy_path, _ = line.split(",")

        noisy_path = fix_path(noisy_path)

        if os.path.exists(noisy_path):
            all_image_paths.append(noisy_path)
        else:
            missing_files.append(noisy_path)

print("\n✅ TOTAL IMAGES:", len(all_image_paths))
if missing_files:
    print(f"⚠️  MISSING FILES: {len(missing_files)}")
    for mf in missing_files[:5]:
        print(f"   - {mf}")
    if len(missing_files) > 5:
        print(f"   ... and {len(missing_files) - 5} more")

🔍 Building dataset paths...


📂 Dataset Files: 100%|██████████| 7/7 [00:00<00:00, 27.41it/s]


✅ TOTAL IMAGES: 90752


In [14]:
BASE_SAVE = "ckpt/Phase3/BLIP_CLIP_degradation_extractor_Classifier/Captions"

os.makedirs(BASE_SAVE, exist_ok=True)

CSV_PATH   = os.path.join(BASE_SAVE, "captions_dataset.csv")
CACHE_PATH = os.path.join(BASE_SAVE, "captions_cache.pt")
STATE_PATH = os.path.join(BASE_SAVE, "captions_state.pt")

SAVE_EVERY = 1000
BATCH_SIZE = 48

In [15]:
class DatasetManager:
    def __init__(self, csv_path, cache_path, state_path):

        self.csv_path = csv_path
        self.cache_path = cache_path
        self.state_path = state_path

        self.df = pd.DataFrame(columns=["image_path", "caption"])
        self.cache = {}
        self.state = {"processed": set()}

        self._load_all()

    def _load_all(self):

        # CSV
        if os.path.exists(self.csv_path):
            self.df = pd.read_csv(self.csv_path)
            print(f"[CSV] Loaded {len(self.df)} rows")

        # CACHE
        if os.path.exists(self.cache_path):
            self.cache = torch.load(self.cache_path)
            print(f"[CACHE] Loaded {len(self.cache)} items")

        # STATE
        if os.path.exists(self.state_path):
            self.state = torch.load(self.state_path)
            print(f"[STATE] Resumed {len(self.state['processed'])} items")

        # sync CSV → cache
        for _, row in self.df.iterrows():
            self.cache[row["image_path"]] = row["caption"]

    def add_batch(self, batch_data):
        """
        batch_data: list of (path, caption)
        """
        df_new = pd.DataFrame(batch_data, columns=["image_path", "caption"])
        self.df = pd.concat([self.df, df_new], ignore_index=True)

        for p, c in batch_data:
            self.cache[p] = c
            self.state["processed"].add(p)

    def exists(self, path):
        return path in self.cache or path in self.state["processed"]

    def save(self):
        self.df.to_csv(self.csv_path, index=False)
        torch.save(self.cache, self.cache_path)
        torch.save(self.state, self.state_path)

manager = DatasetManager(CSV_PATH, CACHE_PATH, STATE_PATH)

In [16]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
base_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16
).to(device)
base_model.eval()

Loading weights: 100%|██████████| 1247/1247 [00:02<00:00, 486.32it/s]


Blip2ForConditionalGeneration(
  (vision_model): Blip2VisionModel(
    (embeddings): Blip2VisionEmbeddings(
      (patch_embedding): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
    )
    (encoder): Blip2Encoder(
      (layers): ModuleList(
        (0-38): 39 x Blip2EncoderLayer(
          (self_attn): Blip2Attention(
            (qkv): Linear(in_features=1408, out_features=4224, bias=True)
            (projection): Linear(in_features=1408, out_features=1408, bias=True)
          )
          (layer_norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): Blip2MLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=1408, out_features=6144, bias=True)
            (fc2): Linear(in_features=6144, out_features=1408, bias=True)
          )
          (layer_norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True, bias=True)
        )
      )
    )
    (post_layernorm): LayerNorm((1408,), eps=1e-06, elementwise_

In [20]:
remaining_paths = [p for p in all_image_paths if not manager.exists(p)]

print("TOTAL:", len(all_image_paths))
print("REMAINING:", len(remaining_paths))

buffer = []
failed_paths = []

for i in tqdm(range(0, len(remaining_paths), BATCH_SIZE), desc="Captioning"):

    batch_paths = remaining_paths[i:i+BATCH_SIZE]

    try:
        results = generate_caption_batch_with_degradation(
            batch_paths,
            base_model,
            processor,
            classifier_model,
            device
        )

        # results already contains (path, caption)
        buffer.extend(results)

    except Exception as e:
        print(f"\n❌ Batch failed at index {i}: {e}")
        failed_paths.extend(batch_paths)
        continue

    if len(buffer) >= SAVE_EVERY:
        manager.add_batch(buffer)
        manager.save()
        print(f"Saved {len(manager.df)} captions")
        buffer.clear()

# Save remaining captions
if buffer:
    manager.add_batch(buffer)
    manager.save()
    print(f"Final save: {len(manager.df)} captions")

TOTAL: 90752
REMAINING: 90752


Captioning:   1%|          | 21/1891 [00:59<1:31:52,  2.95s/it]

Saved 1008 captions


Captioning:   2%|▏         | 42/1891 [01:59<1:32:33,  3.00s/it]

Saved 2016 captions


...

Saved 89712 captions


Captioning: 100%|█████████▉| 1890/1891 [1:06:46<00:02,  2.89s/it]

Saved 90720 captions


Captioning: 100%|██████████| 1891/1891 [1:06:48<00:00,  2.12s/it]


Final save: 90752 captions


In [21]:
print("ALL IMAGES:", len(all_image_paths))
print("ALREADY DONE:", len(manager.state["processed"]))
print("TO PROCESS:", len([p for p in all_image_paths if not manager.exists(p)]))

if failed_paths:
    print(f"\n⚠️  FAILED PATHS: {len(failed_paths)}")
    for fp in failed_paths[:5]:
        print(f"   - {fp}")
    if len(failed_paths) > 5:
        print(f"   ... and {len(failed_paths) - 5} more")

ALL IMAGES: 90752
ALREADY DONE: 90752
TO PROCESS: 0


In [22]:
if len(buffer) > 0:
    manager.add_batch(buffer)
    manager.save()

print("\nDONE")
print("Total captions:", len(manager.df))


DONE
Total captions: 90784


In [23]:
clip_model, preprocess, _ = open_clip.create_model_and_transforms(
    'ViT-B-32',
    pretrained='laion2b_s34b_b79k'
)

model = DA_adapter(clip_model).to(device)
model.train()


for name, param in model.named_parameters():
    if "adapter" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.requires_grad for p in model.parameters())
print("Trainable params:", trainable)

Trainable params: 24


In [24]:
class FastDataset(Dataset):
    def __init__(self, image_paths, captions_cache, preprocess):
        self.image_paths = image_paths
        self.captions_cache = captions_cache
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]

        image = Image.open(path).convert("RGB")
        image = self.preprocess(image)

        caption = self.captions_cache[path]

        return image, caption

In [25]:
captions_cache = torch.load(CACHE_PATH, map_location="cpu")
print("Loaded captions:", len(captions_cache))

train_image_paths = [
    p for p in all_image_paths
    if p in captions_cache
]

print("Train images:", len(train_image_paths))

Loaded captions: 90752
Train images: 90752


In [26]:
dataset = FastDataset(
    train_image_paths,
    captions_cache,
    preprocess
)

train_loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=12,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

val_loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=12,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)


In [27]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=args.learning_rate,
    betas=(0.9, 0.98),
    weight_decay=0.01
)

criterion = torch.nn.CrossEntropyLoss()
criterion = nn.CrossEntropyLoss()

temperature = 0.07

In [29]:
# =====================================================
# TEST ONE CAPTION + TOKENIZATION
# =====================================================

print("Caption cache loaded:", len(captions_cache))

sample_path = all_image_paths[0]

print("=" * 80)
print("SAMPLE IMAGE PATH:")
print(sample_path)

print("\nCLASSIFIER ENHANCE GENERATED CAPTION:")
print(captions_cache[sample_path])

tokens = tokenizer([captions_cache[sample_path]])

print("\nTOKEN TENSOR SHAPE:")
print(tokens.shape)

print("\nTOKEN IDS:")
print(tokens)

print("\nTOKEN IDS (LIST FORMAT):")
print(tokens[0].tolist())

print("=" * 80)

Caption cache loaded: 90752
SAMPLE IMAGE PATH:
datasets/denoising_datasets/BSD400/noisy15/test_001.png

CLASSIFIER ENHANCE GENERATED CAPTION:
A bear and her cubs in heavy noise conditions

TOKEN TENSOR SHAPE:
torch.Size([1, 77])

TOKEN IDS:
tensor([[49406,   320,  4298,   537,   899,  9858,   530,  4200,  9307,  5892,
         49407,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0]])

TOKEN IDS (LIST FORMAT):
[49406, 320, 4298, 537, 899, 9858, 530, 4200, 9307, 5892, 49407, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [30]:
start_epoch = args.start_epoch

DRIVE_SAVE_DIR = args.checkpoints_dir

os.makedirs(
    DRIVE_SAVE_DIR,
    exist_ok=True
)

latest_ckpt_path = os.path.join(
    DRIVE_SAVE_DIR,
    "model_latest.pth"
)
best_ckpt_path = os.path.join(
    DRIVE_SAVE_DIR,
    "model_best.pth"
)

def load_checkpoint(ckpt_path):
    print(f"[INFO] Loading checkpoint: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])

    if "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    elif "optimizer" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer"])
    else:
        print("[WARNING] Optimizer state not found in checkpoint.")

    return checkpoint["epoch"] + 1

if os.path.exists(latest_ckpt_path):
    try:
        start_epoch = load_checkpoint(latest_ckpt_path)
        print(f"[INFO] Resume Epoch {start_epoch} from latest")
    except Exception as e:
        print(f"[ERROR] Failed to load latest checkpoint: {e}")
        if os.path.exists(best_ckpt_path):
            try:
                print("[INFO] Attempting to load best checkpoint instead...")
                start_epoch = load_checkpoint(best_ckpt_path)
                print(f"[INFO] Resume Epoch {start_epoch} from best")
            except Exception as e2:
                print(f"[ERROR] Failed to load best checkpoint as well: {e2}")
                print("[INFO] Training From Scratch")
        else:
            print("[INFO] Training From Scratch")
else:
    print("[INFO] Training From Scratch")


[INFO] Training From Scratch


In [31]:
try:
    # Re-enable gradient tracking that was turned off during BLIP-2 loading
    torch.set_grad_enabled(True)

    # Get one batch from the training loader to define image_features and text_features
    images, captions = next(iter(train_loader))

    images = images.to(device, non_blocking=True)
    texts = tokenizer(list(captions)).to(device)

    # Ensure model is in training mode to track gradients
    model.train()

    # Perform a forward pass
    image_features, text_features = model(images, texts)

    # Calculate a dummy loss to define the 'loss' variable
    image_features_norm = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features_norm = text_features / text_features.norm(dim=-1, keepdim=True)
    logits = (image_features_norm @ text_features_norm.T) / temperature
    targets = torch.arange(len(images), device=device)
    loss = criterion(logits, targets)

    print("image requires_grad:", image_features.requires_grad)
    print("text requires_grad:", text_features.requires_grad)
    print("loss requires_grad:", loss.requires_grad)
except NameError as e:
    print(f"Error: {e}. Please ensure `train_loader`, `tokenizer`, `model`, `device`, `temperature`, and `criterion` are defined and accessible.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

image requires_grad: True
text requires_grad: True
loss requires_grad: True


In [32]:
# Re-enable gradient tracking
torch.set_grad_enabled(True)

train_losses = []
train_accs = []

val_losses = []
val_accs = []

best_val_acc = 0.0

for epoch in range(start_epoch, args.end_epoch):

    # TRAINING
    model.train()

    epoch_loss = 0.0
    epoch_correct = 0
    epoch_total = 0

    pbar = tqdm(train_loader, desc=f"Train Epoch {epoch}")

    for images, captions in pbar:

        images = images.to(device, non_blocking=True)
        texts = tokenizer(list(captions)).to(device)

        # Forward
        image_features, text_features = model(images, texts)

        image_features = image_features / image_features.norm(
            dim=-1, keepdim=True
        )
        text_features = text_features / text_features.norm(
            dim=-1, keepdim=True
        )

        logits = 100.0 * (image_features @ text_features.T)

        targets = torch.arange(len(images), device=device)

        loss = criterion(logits, targets)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        preds = logits.argmax(dim=1)
        batch_correct = (preds == targets).sum().item()

        epoch_correct += batch_correct
        epoch_total += len(images)

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{100 * batch_correct / len(images):.1f}%"
        })

    train_loss = epoch_loss / len(train_loader)
    train_acc = epoch_correct / epoch_total

    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # VALIDATION

    model.eval()

    val_epoch_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        pbar_val = tqdm(val_loader, desc=f"Val Epoch {epoch}")

        for images, captions in pbar_val:

            images = images.to(device, non_blocking=True)
            texts = tokenizer(list(captions)).to(device)

            image_features, text_features = model(images, texts)

            image_features = image_features / image_features.norm(
                dim=-1, keepdim=True
            )
            text_features = text_features / text_features.norm(
                dim=-1, keepdim=True
            )

            logits = 100.0 * (image_features @ text_features.T)

            targets = torch.arange(len(images), device=device)

            loss = criterion(logits, targets)

            val_epoch_loss += loss.item()

            preds = logits.argmax(dim=1)

            val_correct += (preds == targets).sum().item()
            val_total += len(images)

            pbar_val.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{100 * (preds == targets).sum().item() / len(images):.1f}%"
            })

    val_loss = val_epoch_loss / len(val_loader)
    val_acc = val_correct / val_total

    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # PRINT METRICS
    print(
        f"\nEpoch {epoch} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc : {train_acc*100:.2f}% | "
        f"Val Loss  : {val_loss:.4f} | "
        f"Val Acc   : {val_acc*100:.2f}% | "
    )

    learnable_params = {
        name: param.detach().cpu()
        for name, param in model.named_parameters()
        if param.requires_grad
    }

    # SAVE LATEST CHECKPOINT
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "learnable_params": learnable_params,
        "train_losses": train_losses,
        "train_accs": train_accs,
        "val_losses": val_losses,
        "val_accs": val_accs,
        "best_val_acc": best_val_acc,
    }

    torch.save(checkpoint, latest_ckpt_path)

    # SAVE BEST CHECKPOINT
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        checkpoint["best_val_acc"] = best_val_acc

        torch.save(checkpoint, best_ckpt_path)

        print(
            f"[BEST] New best model saved "
            f"(Val Acc = {best_val_acc*100:.2f}%)"
        )

    print(
        f"[LATEST] Checkpoint saved "
        f"(Epoch {epoch})"
    )

Val Epoch 0: 100%|██████████| 1418/1418 [03:20<00:00,  7.08it/s, loss=1.4463, acc=57.8%]



Epoch 0 | Train Loss: 1.8129 | Train Acc : 47.12% | Val Loss  : 0.9203 | Val Acc   : 68.92% | 
[BEST] New best model saved (Val Acc = 68.92%)
[LATEST] Checkpoint saved (Epoch 0)


Val Epoch 1: 100%|██████████| 1418/1418 [03:20<00:00,  7.08it/s, loss=0.7767, acc=73.4%]



Epoch 1 | Train Loss: 0.7275 | Train Acc : 74.86% | Val Loss  : 0.5550 | Val Acc   : 80.26% | 
[BEST] New best model saved (Val Acc = 80.26%)
[LATEST] Checkpoint saved (Epoch 1)


Val Epoch 2: 100%|██████████| 1418/1418 [03:19<00:00,  7.10it/s, loss=0.4993, acc=84.4%]


...


Epoch 97 | Train Loss: 0.0880 | Train Acc : 95.56% | Val Loss  : 0.0848 | Val Acc   : 95.70% | 
[BEST] New best model saved (Val Acc = 95.70%)
[LATEST] Checkpoint saved (Epoch 97)


Val Epoch 98: 100%|██████████| 1418/1418 [02:47<00:00,  8.48it/s, loss=0.1616, acc=89.1%] 



Epoch 98 | Train Loss: 0.0884 | Train Acc : 95.55% | Val Loss  : 0.0855 | Val Acc   : 95.67% | 
[LATEST] Checkpoint saved (Epoch 98)


Val Epoch 99: 100%|██████████| 1418/1418 [02:47<00:00,  8.47it/s, loss=0.0516, acc=96.9%] 



Epoch 99 | Train Loss: 0.0887 | Train Acc : 95.50% | Val Loss  : 0.0882 | Val Acc   : 95.52% | 
[LATEST] Checkpoint saved (Epoch 99)


In [33]:
!mkdir -p blip_vlunet_pretrained_ckpt
!cp -r ckpt/Phase3/BLIP_CLIP_degradation_extractor_Classifier/Checkpoint/model_best.pth \
    blip_vlunet_pretrained_ckpt/blip_vlunet_class_clip_tuned.pth